# Heston Model Volatility Surface Arbitrage

## A Step-by-Step Implementation

This notebook implements a complete pipeline:
1. **Data Ingestion** — Download options chain data via `yfinance`
2. **Surface Construction** — Build the implied volatility surface
3. **Heston Calibration** — Fit Heston model parameters to the surface
4. **Mispricing Detection** — Find strikes where market IV deviates from Heston fair value
5. **Strategy Generation** — Construct delta-hedged straddles/strangles
6. **Backtest & P&L Attribution** — Track vol capture, delta drift, gamma scalping

---
**Requirements:** numpy, pandas, scipy, yfinance, py_vollib, matplotlib, plotly

---
## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.optimize import differential_evolution, minimize
from scipy.stats import norm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Optional: py_vollib for Black-Scholes IV
try:
    from py_vollib.black_scholes import black_scholes as bs_price
    from py_vollib.black_scholes.implied_volatility import implied_volatility as bs_iv
    HAVE_PY_VOLLIB = True
except ImportError:
    HAVE_PY_VOLLIB = False
    print('py_vollib not installed - will use approximate IV from notebook')

try:
    import yfinance as yf
    HAVE_YFINANCE = True
except ImportError:
    HAVE_YFINANCE = False
    print('yfinance not installed - will use synthetic data')

# Try plotly for 3D surfaces
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False

print('All imports loaded successfully.')

In [ ]:
# --- CONFIGURATION ---
TICKER = 'SPY'
RISK_FREE_RATE = 0.05      # Approximate short rate
DIVIDEND_YIELD = 0.013     # SPY approximate dividend yield
CALIBRATION_THRESHOLD = 2.0  # Z-score threshold for mispricing
MIN_VOLUME = 10             # Minimum option volume to consider
MIN_BID = 0.05              # Minimum bid price to filter illiquid options

# Heston parameter bounds for optimization
BOUNDS = {
    'kappa': (0.1, 10.0),     # Mean reversion speed
    'theta': (0.005, 0.20),   # Long-term variance
    'xi':    (0.05, 1.0),     # Vol of vol
    'rho':   (-0.98, 0.98),   # Correlation
    'v0':    (0.005, 0.20),   # Initial variance
}

print(f'Configuration set for {TICKER}')

---
## 2. Data Ingestion

Download options chains from Yahoo Finance. If yfinance is unavailable, generate a realistic synthetic surface for testing.

In [ ]:
def download_options_chain(ticker, expiration_dates=None):
    """Download options chain for a ticker.
    
    Returns a DataFrame with columns:
        expiry, strike, option_type, bid, ask, lastPrice, 
        impliedVol, volume, openInterest, underlying_price
    """
    if not HAVE_YFINANCE:
        print('yfinance unavailable - using synthetic data')
        return None
    
    try:
        stock = yf.Ticker(ticker)
        current_price = stock.history(period='1d')['Close'].iloc[-1]
        
        all_chains = []
        exps = stock.options
        if expiration_dates is not None:
            exps = [e for e in exps if e in expiration_dates]
        
        # Limit to first 5 expiries to keep calibration reasonable
        for expiry in exps[:5]:
            opt_chain = stock.option_chain(expiry)
            for opt_type, df in [('call', opt_chain.calls), ('put', opt_chain.puts)]:
                df = df.copy()
                df['option_type'] = opt_type
                df['expiry'] = expiry
                all_chains.append(df)
        
        if not all_chains:
            return None
        
        full = pd.concat(all_chains, ignore_index=True)
        full['underlying_price'] = current_price
        
        # Filter out illiquid options
        full = full[(full['volume'] > MIN_VOLUME) | (full['openInterest'] > 0)]
        full = full[full['bid'] > MIN_BID]
        
        print(f'Downloaded {len(full)} options for {ticker} across {len(exps[:5])} expiries')
        print(f'Underlying price: ${current_price:.2f}')
        return full
    
    except Exception as e:
        print(f'Error downloading options: {e}')
        return None

### Fallback: Synthetic Data Generator

Generates a realistic options surface using a known Heston parameter set + noise.

In [ ]:
def bs_implied_vol(price, S, K, T, r, q, flag):
    """Compute Black-Scholes implied vol from option price."""
    if HAVE_PY_VOLLIB:
        try:
            iv = bs_iv(price, S, K, T, r, flag.upper()[0])
            return max(iv, 0.001)
        except:
            return np.nan
    
    # Fallback: Newton-Raphson for IV
    def bs_price_sigma(sigma):
        d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        if flag.lower() == 'call':
            return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            return K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
    
    sigma_guess = 0.3
    for _ in range(100):
        p = bs_price_sigma(sigma_guess)
        vega = S * np.exp(-q * T) * np.sqrt(T) * norm.pdf(
            (np.log(S / K) + (r - q + 0.5 * sigma_guess**2) * T) / (sigma_guess * np.sqrt(T))
        )
        diff = p - price
        if abs(diff) < 1e-8:
            break
        sigma_guess -= diff / (vega + 1e-10)
        sigma_guess = max(sigma_guess, 0.001)
    return sigma_guess


def generate_synthetic_surface():
    """Generate a synthetic options surface from known Heston parameters + noise."""
    np.random.seed(42)
    
    # True Heston parameters used to generate data
    true_params = {
        'kappa': 2.5,
        'theta': 0.04,   # 20% long-term vol
        'xi': 0.3,
        'rho': -0.7,
        'v0': 0.0225,    # 15% initial vol
    }
    
    S = 500.0  # SPY-like
    r = RISK_FREE_RATE
    q = DIVIDEND_YIELD
    
    expiries_days = [30, 60, 91, 182, 365]
    strikes = np.arange(420, 590, 10)
    
    rows = []
    for T_days in expiries_days:
        T = T_days / 365.0
        
        # Compute ATM IV for this expiry to serve as reference
        atm_iv = np.sqrt(true_params['v0'] * np.exp(-true_params['kappa'] * T) + 
                       true_params['theta'] * (1 - np.exp(-true_params['kappa'] * T)))
        
        for K in strikes:
            moneyness = np.log(K / (S * np.exp((r - q) * T)))
            
            # Generate IV with smile using a simplified approximation
            iv = atm_iv + 0.1 * moneyness**2 + 0.3 * moneyness
            iv = max(iv, 0.01)
            
            # Add noise to simulate market bid-ask
            iv_noise = iv + np.random.normal(0, 0.005)
            iv_noise = max(iv_noise, 0.01)
            
            for flag in ['call', 'put']:
                try:
                    # Price using BS with the noisy IV
                    d1 = (np.log(S / K) + (r - q + 0.5 * iv**2) * T) / (iv * np.sqrt(T))
                    d2 = d1 - iv * np.sqrt(T)
                    
                    if flag == 'call':
                        price = S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
                        delta = np.exp(-q * T) * norm.cdf(d1)
                        gamma = np.exp(-q * T) * norm.pdf(d1) / (S * iv * np.sqrt(T))
                        theta = -S * np.exp(-q * T) * norm.pdf(d1) * iv / (2 * np.sqrt(T)) \
                                - r * K * np.exp(-r * T) * norm.cdf(d2) + q * S * np.exp(-q * T) * norm.cdf(d1)
                    else:
                        price = K * np.exp(-r * T) * norm.cdf(-d2) - S * np.exp(-q * T) * norm.cdf(-d1)
                        delta = np.exp(-q * T) * (norm.cdf(d1) - 1)
                        gamma = np.exp(-q * T) * norm.pdf(d1) / (S * iv * np.sqrt(T))
                        theta = -S * np.exp(-q * T) * norm.pdf(d1) * iv / (2 * np.sqrt(T)) \
                                + r * K * np.exp(-r * T) * norm.cdf(-d2) - q * S * np.exp(-q * T) * norm.cdf(-d1)
                    
                    vega_val = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)
                    
                    bid = price * 0.97
                    ask = price * 1.03
                    
                    rows.append({
                        'expiry': (datetime.now() + timedelta(days=T_days)).strftime('%Y-%m-%d'),
                        'strike': K,
                        'option_type': flag,
                        'bid': bid,
                        'ask': ask,
                        'lastPrice': price,
                        'impliedVol': iv_noise,
                        'volume': np.random.randint(100, 50000),
                        'openInterest': np.random.randint(500, 200000),
                        'underlying_price': S,
                        'delta': delta,
                        'gamma': gamma,
                        'theta': theta,
                        'vega': vega_val,
                    })
                except Exception:
                    pass
    
    df = pd.DataFrame(rows)
    print(f'Generated {len(df)} synthetic options')
    print(f'Underlying price: ${S:.2f}')
    print(f'True params (used to generate): {true_params}')
    return df, true_params

In [ ]:
# --- LOAD DATA ---

data = download_options_chain(TICKER)
true_params_known = None

if data is None or len(data) < 10:
    print('Falling back to synthetic data...')
    data, true_params_known = generate_synthetic_surface()

print(f'\nData shape: {data.shape}')
data.head()

---
## 3. Surface Construction

Build a 2D implied volatility surface (strike x time-to-expiry).
We use OTM options (calls for strikes > forward, puts for strikes < forward) because they are more liquid.

In [ ]:
def build_iv_surface(df):
    """Construct a clean IV surface DataFrame using OTM options."""
    df = df.copy()
    
    # Parse expiry and compute TTE
    df['expiry_dt'] = pd.to_datetime(df['expiry'])
    ref_date = pd.Timestamp.today()
    df['TTE'] = (df['expiry_dt'] - ref_date).dt.days / 365.0
    df['TTE'] = df['TTE'].clip(lower=1/365)
    
    S = df['underlying_price'].iloc[0]
    r = RISK_FREE_RATE
    q = DIVIDEND_YIELD
    
    # Forward price and moneyness
    df['forward'] = S * np.exp((r - q) * df['TTE'])
    df['moneyness'] = df['strike'] / df['forward']
    
    # Use mid-price IV
    df['mid'] = (df['bid'] + df['ask']) / 2
    df['mid_iv'] = df['impliedVol']
    
    # Select OTM options for the surface
    otm_calls = (df['option_type'] == 'call') & (df['strike'] > df['forward'])
    otm_puts = (df['option_type'] == 'put') & (df['strike'] < df['forward'])
    surface_df = df[otm_calls | otm_puts].copy()
    
    print(f'Surface built: {len(surface_df)} OTM options across {surface_df["expiry"].nunique()} expiries')
    return surface_df


surface = build_iv_surface(data)
print(f'\nSample surface data:')
surface[['expiry', 'strike', 'option_type', 'mid_iv', 'TTE', 'moneyness']].head(10)

In [ ]:
# --- VISUALIZE IV SURFACE ---

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

expiry_groups = surface.groupby('expiry')
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(expiry_groups)))

for (exp, group), color in zip(expiry_groups, colors):
    tte = group['TTE'].iloc[0]
    ax.scatter(group['moneyness'], tte, group['mid_iv'], 
               c=[color], label=f'T={tte*365:.0f}d', alpha=0.7, s=30)

ax.set_xlabel('Moneyness (K/F)')
ax.set_ylabel('Time to Expiry (years)')
ax.set_zlabel('Implied Volatility')
ax.set_title(f'{TICKER} Implied Volatility Surface')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## 4. Heston Model Implementation

We implement the Heston pricing formula using:
- Albrecher et al. (2007) formulation to avoid the "Little Heston Trap"
- Gauss-Laguerre quadrature for the Fourier inversion
- Conversion from model price to Black-Scholes implied volatility

In [ ]:
def heston_char_func(u, kappa, theta, xi, rho, v0, T):
    """
    Heston characteristic function - Albrecher et al. (2007) formulation.
    This avoids branch cut issues from the original Heston (1993) paper.
    """
    i = 1j
    
    d = np.sqrt((rho * xi * i * u - kappa)**2 - xi**2 * (-i * u - u**2))
    g = (kappa - rho * xi * i * u - d) / (kappa - rho * xi * i * u + d)
    
    exp_dT = np.exp(-d * T)
    
    C = kappa * theta / xi**2 * ((kappa - rho * xi * i * u - d) * T - 2 * np.log((1 - g * exp_dT) / (1 - g)))
    D = (kappa - rho * xi * i * u - d) / xi**2 * ((1 - exp_dT) / (1 - g * exp_dT))
    
    return np.exp(C + D * v0)


def heston_call_price(S, K, T, r, q, kappa, theta, xi, rho, v0):
    """
    Compute Heston call option price using Lewis (2001) Fourier inversion.
    Uses Gauss-Laguerre quadrature for efficiency.
    """
    F = S * np.exp((r - q) * T)
    x = np.log(F / K)
    
    def integrand(u):
        if abs(u) < 1e-12:
            return 0.0
        phi = heston_char_func(u - 0.5j, kappa, theta, xi, rho, v0, T)
        numerator = phi * np.exp(-0.5j * u * x + 0.5 * x)
        denominator = u**2 + 0.25
        return (numerator / denominator).real
    
    # Gauss-Laguerre nodes and weights (n=64 for accuracy)
    nodes, weights = np.polynomial.laguerre.laggauss(64)
    
    integral = np.sum(weights * integrand(nodes))
    
    call_price = F * np.exp(-r * T) - K * np.exp(-r * T) * np.sqrt(F * K) * np.exp(-x**2 / 2) * integral / np.pi
    
    return max(call_price, 0.0)


def heston_price(S, K, T, r, q, kappa, theta, xi, rho, v0, flag='call'):
    """Compute Heston option price for call or put using put-call parity."""
    call_price = heston_call_price(S, K, T, r, q, kappa, theta, xi, rho, v0)
    
    if flag.lower() == 'call':
        return call_price
    else:
        put_price = call_price - S * np.exp(-q * T) + K * np.exp(-r * T)
        return max(put_price, 0.0)


def heston_iv(S, K, T, r, q, kappa, theta, xi, rho, v0, flag='call'):
    """Compute Heston-implied Black-Scholes IV."""
    price = heston_price(S, K, T, r, q, kappa, theta, xi, rho, v0, flag)
    iv = bs_implied_vol(price, S, K, T, r, q, flag)
    return iv, price


# Quick test
S_test = 500.0
K_test = 500.0
T_test = 0.25
price = heston_call_price(S_test, K_test, T_test, RISK_FREE_RATE, DIVIDEND_YIELD, 
                          2.0, 0.04, 0.3, -0.7, 0.04)
iv_test = bs_implied_vol(price, S_test, K_test, T_test, RISK_FREE_RATE, DIVIDEND_YIELD, 'call')
print(f'Heston ATM call price: ${price:.4f}')
print(f'Heston ATM implied vol: {iv_test:.4f} ({iv_test*100:.2f}%)')
print('Heston pricing engine ready.')

---
## 5. Heston Calibration

The core optimization problem:

$$\min_{\Psi} \sum_{i} \sum_{j} \left(IV_{\text{mkt}}(T_i, K_j) - IV_{\text{Heston}}(T_i, K_j; \Psi)\right)^2$$

We use `scipy.optimize.differential_evolution` for global optimization, followed by L-BFGS-B for local refinement.

In [ ]:
def calibration_loss(params, surface_df, S, r, q):
    """Objective function for Heston calibration. Returns RMSE between market and Heston IVs."""
    kappa, theta, xi, rho, v0 = params
    
    # Enforce Feller condition as soft penalty (2*kappa*theta > xi^2)
    feller_penalty = max(0, xi**2 - 2 * kappa * theta) * 10
    
    errors = []
    for _, row in surface_df.iterrows():
        try:
            heston_iv_val, _ = heston_iv(
                S, row['strike'], row['TTE'], r, q,
                kappa, theta, xi, rho, v0,
                row['option_type']
            )
            if not np.isnan(heston_iv_val) and not np.isnan(row['mid_iv']):
                errors.append((heston_iv_val - row['mid_iv']) ** 2)
        except:
            pass
    
    if len(errors) == 0:
        return 1.0
    
    rmse = np.sqrt(np.mean(errors))
    return rmse + feller_penalty


def calibrate_heston(surface_df, S, r, q):
    """
    Calibrate Heston model to the IV surface.
    Uses Differential Evolution (global) + L-BFGS-B (local).
    """
    bounds = [
        BOUNDS['kappa'],
        BOUNDS['theta'],
        BOUNDS['xi'],
        BOUNDS['rho'],
        BOUNDS['v0'],
    ]
    
    print('Stage 1: Differential Evolution (global search)...')
    print(f'Calibrating to {len(surface_df)} options...')
    
    result_de = differential_evolution(
        calibration_loss,
        bounds=bounds,
        args=(surface_df, S, r, q),
        maxiter=100,
        popsize=15,
        tol=1e-6,
        mutation=(0.5, 1.0),
        recombination=0.9,
        seed=42,
        polish=True,
    )
    
    print(f'  DE result: RMSE={result_de.fun:.6f}')
    print(f'  Params: kappa={result_de.x[0]:.4f}, theta={result_de.x[1]:.4f}, '
          f'xi={result_de.x[2]:.4f}, rho={result_de.x[3]:.4f}, v0={result_de.x[4]:.4f}')
    
    # Stage 2: Local refinement
    print('\nStage 2: L-BFGS-B (local refinement)...')
    result_local = minimize(
        calibration_loss,
        x0=result_de.x,
        args=(surface_df, S, r, q),
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 500, 'ftol': 1e-10}
    )
    
    print(f'  Local result: RMSE={result_local.fun:.6f}')
    print(f'  Params: kappa={result_local.x[0]:.4f}, theta={result_local.x[1]:.4f}, '
          f'xi={result_local.x[2]:.4f}, rho={result_local.x[3]:.4f}, v0={result_local.x[4]:.4f}')
    
    params = {
        'kappa': result_local.x[0],
        'theta': result_local.x[1],
        'xi': result_local.x[2],
        'rho': result_local.x[3],
        'v0': result_local.x[4],
    }
    
    return params, result_local.fun

In [ ]:
# --- RUN CALIBRATION ---

S = data['underlying_price'].iloc[0]
r = RISK_FREE_RATE
q = DIVIDEND_YIELD

print('=' * 60)
print('CALIBRATING HESTON MODEL')
print('=' * 60)
print(f'Underlying price: ${S:.2f}')
print(f'Risk-free rate: {r*100:.1f}%')
print(f'Dividend yield: {q*100:.1f}%')
print(f'Number of options in surface: {len(surface)}')

heston_params, calib_rmse = calibrate_heston(surface, S, r, q)

print('\n' + '=' * 60)
print('CALIBRATION COMPLETE')
print('=' * 60)

if true_params_known:
    print('\nComparison with true parameters:')
    for k in heston_params:
        print(f'  {k}: estimated={heston_params[k]:.4f}, true={true_params_known[k]:.4f}')

print(f'\nFinal calibration RMSE: {calib_rmse:.6f} ({calib_rmse*10000:.1f} basis points)')

---
## 6. Calibration Quality Check

Plot market IV vs. Heston fitted IV to visually inspect fit quality.

In [ ]:
def compute_fitted_surface(surface_df, params, S, r, q):
    """Compute Heston IV for every point in the surface."""
    result = surface_df.copy()
    fitted_ivs = []
    
    for _, row in result.iterrows():
        try:
            iv_val, _ = heston_iv(
                S, row['strike'], row['TTE'], r, q,
                params['kappa'], params['theta'], params['xi'],
                params['rho'], params['v0'], row['option_type']
            )
            fitted_ivs.append(iv_val)
        except:
            fitted_ivs.append(np.nan)
    
    result['heston_iv'] = fitted_ivs
    result['iv_residual'] = result['mid_iv'] - result['heston_iv']
    return result


fitted_surface = compute_fitted_surface(surface, heston_params, S, r, q)

# Plot residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Market IV vs Heston IV
ax = axes[0]
ax.scatter(fitted_surface['mid_iv'], fitted_surface['heston_iv'], 
           alpha=0.5, s=20, c=fitted_surface['TTE'], cmap='viridis')
min_val = min(fitted_surface['mid_iv'].min(), fitted_surface['heston_iv'].min())
max_val = max(fitted_surface['mid_iv'].max(), fitted_surface['heston_iv'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=1, label='Perfect fit')
ax.set_xlabel('Market IV')
ax.set_ylabel('Heston IV')
ax.set_title('Market vs Heston Implied Volatility')
ax.legend()
ax.grid(True, alpha=0.3)

# Residual histogram
ax = axes[1]
residuals = fitted_surface['iv_residual'].dropna()
ax.hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(0, color='red', linestyle='--', lw=1)
ax.set_xlabel('IV Residual (Market - Heston)')
ax.set_ylabel('Frequency')
ax.set_title(f'Residual Distribution\nMean={residuals.mean():.4f}, Std={residuals.std():.4f}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Residuals: mean={residuals.mean():.6f}, std={residuals.std():.6f}')
print(f'RMSE: {np.sqrt(np.mean(residuals**2)):.6f}')

---
## 7. Mispricing Detection

Identify strikes where the market IV deviates significantly from the Heston fair value.
We use a z-score approach:

$$z_i = \frac{IV_{mkt,i} - IV_{Heston,i}}{\sigma_{residuals}}$$

In [ ]:
def detect_mispricings(fitted_surface, z_threshold=2.0):
    """
    Detect mispriced options where market IV deviates from Heston IV.
    Returns DataFrame with mispricing signals added.
    """
    df = fitted_surface.copy()
    
    # Compute z-scores per expiry to account for term structure
    df['z_score'] = np.nan
    
    for expiry in df['expiry'].unique():
        mask = df['expiry'] == expiry
        residuals = df.loc[mask, 'iv_residual']
        std_resid = residuals.std()
        if std_resid > 0:
            df.loc[mask, 'z_score'] = residuals / std_resid
    
    # Flag mispricings
    df['is_mispriced'] = abs(df['z_score']) > z_threshold
    df['mispricing_type'] = 'fair'
    df.loc[(df['z_score'] > z_threshold), 'mispricing_type'] = 'overpriced'
    df.loc[(df['z_score'] < -z_threshold), 'mispricing_type'] = 'underpriced'
    
    return df


mispricing_df = detect_mispricings(fitted_surface, z_threshold=CALIBRATION_THRESHOLD)

print('Mispricing detection results:')
print(f'  Total options: {len(mispricing_df)}')
print(f'  Overpriced (market IV too high): {(mispricing_df["mispricing_type"] == "overpriced").sum()}')
print(f'  Underpriced (market IV too low): {(mispricing_df["mispricing_type"] == "underpriced").sum()}')
print(f'  Fair: {(mispricing_df["mispricing_type"] == "fair").sum()}')

# Show most mispriced options
mispriced = mispricing_df[mispricing_df['is_mispriced']].sort_values('z_score', key=abs, ascending=False)
if len(mispriced) > 0:
    print('\nTop 10 most mispriced options:')
    display(mispriced[[
        'expiry', 'strike', 'option_type', 'mid_iv', 'heston_iv',
        'iv_residual', 'z_score', 'mispricing_type', 'volume'
    ]].head(10))

In [ ]:
# Visualize mispricings across the surface

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Z-scores by moneyness
ax = axes[0]
for exp in mispricing_df['expiry'].unique():
    subset = mispricing_df[mispricing_df['expiry'] == exp]
    tte_days = int(subset['TTE'].iloc[0] * 365)
    ax.scatter(subset['moneyness'], subset['z_score'], 
               label=f'{tte_days}d', alpha=0.6, s=20)
ax.axhline(CALIBRATION_THRESHOLD, color='r', linestyle='--', alpha=0.5, label='Threshold')
ax.axhline(-CALIBRATION_THRESHOLD, color='r', linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
ax.set_xlabel('Moneyness (K/F)')
ax.set_ylabel('Z-Score')
ax.set_title('Mispricing Z-Scores Across Surface')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Residuals vs moneyness
ax = axes[1]
colors_map = {'overpriced': 'red', 'underpriced': 'green', 'fair': 'gray'}
for typ, color in colors_map.items():
    subset = mispricing_df[mispricing_df['mispricing_type'] == typ]
    ax.scatter(subset['moneyness'], subset['iv_residual'] * 100, 
               c=color, label=typ, alpha=0.5, s=20)
ax.set_xlabel('Moneyness (K/F)')
ax.set_ylabel('IV Residual (%)')
ax.set_title('IV Residuals by Moneyness')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Strategy Construction

Build delta-hedged straddle positions on mispriced options:
- **Overpriced IV -> Short straddle** (sell options, hedge delta)
- **Underpriced IV -> Long straddle** (buy options, hedge delta)

In [ ]:
def build_trade_signals(mispricing_df, S, r, q):
    """Generate trade signals from mispricing detections."""
    mispriced = mispricing_df[mispricing_df['is_mispriced']].copy()
    
    trades = []
    
    # Group by expiry and direction
    for (expiry, typ), group in mispriced.groupby(['expiry', 'mispricing_type']):
        if len(group) == 0:
            continue
        
        tte = group['TTE'].iloc[0]
        
        direction = -1 if typ == 'overpriced' else 1
        action = 'SELL' if typ == 'overpriced' else 'BUY'
        
        # Find the strike with the largest |z-score| for this expiry/direction
        best = group.loc[group['z_score'].abs().idxmax()]
        K = best['strike']
        
        # Get Heston IV and prices
        call_iv, call_price = heston_iv(S, K, tte, r, q, 
                                        heston_params['kappa'], heston_params['theta'],
                                        heston_params['xi'], heston_params['rho'], 
                                        heston_params['v0'], 'call')
        put_iv, put_price = heston_iv(S, K, tte, r, q,
                                      heston_params['kappa'], heston_params['theta'],
                                      heston_params['xi'], heston_params['rho'], 
                                      heston_params['v0'], 'put')
        
        # Greeks for the straddle (using BS approximation)
        sigma_straddle = (call_iv + put_iv) / 2
        d1 = (np.log(S / K) + (r - q + 0.5 * sigma_straddle**2) * tte) / (sigma_straddle * np.sqrt(tte))
        
        delta_call = np.exp(-q * tte) * norm.cdf(d1)
        delta_put = np.exp(-q * tte) * (norm.cdf(d1) - 1)
        delta_straddle = delta_call + delta_put
        
        gamma = np.exp(-q * tte) * norm.pdf(d1) / (S * sigma_straddle * np.sqrt(tte))
        gamma_straddle = gamma * 2
        
        vega = S * np.exp(-q * tte) * norm.pdf(d1) * np.sqrt(tte)
        vega_straddle = vega * 2
        
        trades.append({
            'expiry': expiry,
            'strike': K,
            'TTE': tte,
            'action': action,
            'direction': direction,
            'mispricing_type': typ,
            'z_score': best['z_score'],
            'market_iv': best['mid_iv'],
            'heston_iv': best['heston_iv'],
            'call_price': call_price * direction,
            'put_price': put_price * direction,
            'straddle_price': (call_price + put_price) * direction,
            'delta_straddle': delta_straddle * direction,
            'gamma_straddle': gamma_straddle * direction,
            'vega_straddle': vega_straddle * direction,
            'hedge_shares': -delta_straddle * direction,
        })
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        trades_df = trades_df.sort_values('z_score', key=abs, ascending=False)
    
    return trades_df


trades = build_trade_signals(mispricing_df, S, r, q)

if len(trades) > 0:
    print(f'Generated {len(trades)} trade signals')
    display(trades[['expiry', 'strike', 'action', 'z_score', 'straddle_price', 'delta_straddle', 'hedge_shares']])
else:
    print('No mispriced options detected at threshold=2.0. Trying lower threshold...')
    mispricing_df = detect_mispricings(fitted_surface, z_threshold=1.0)
    trades = build_trade_signals(mispricing_df, S, r, q)
    print(f'Fallback: generated {len(trades)} trade signals (threshold=1.0)')
    if len(trades) > 0:
        display(trades[['expiry', 'strike', 'action', 'z_score', 'straddle_price', 'delta_straddle', 'hedge_shares']])

---
## 9. Backtest Simulation & P&L Attribution

Simulate a delta-hedged straddle over a holding period and track:
- **Option P&L**: change in straddle value
- **Hedge P&L**: P&L from delta-hedging position
- **Gamma scalping**: convexity benefit from rebalancing

In [ ]:
def simulate_delta_hedged_straddle(trade_row, S0, r, q, params, n_days=5):
    """
    Simulate a delta-hedged straddle over n_days.
    Returns DataFrame with daily P&L breakdown.
    """
    np.random.seed(42)
    
    K = trade_row['strike']
    T = trade_row['TTE']
    direction = trade_row['direction']
    
    kappa = params['kappa']
    theta = params['theta']
    xi = params['xi']
    rho = params['rho']
    v0 = params['v0']
    
    dt = 1/252
    
    results = []
    S = S0
    v = v0
    
    def straddle_value(S_cur, v_cur, t_remaining):
        iv_cur, _ = heston_iv(S_cur, K, t_remaining, r, q,
                              kappa, theta, xi, rho, v_cur, 'call')
        sigma = max(iv_cur, 0.001)
        d1 = (np.log(S_cur / K) + (r - q + 0.5 * sigma**2) * t_remaining) / (sigma * np.sqrt(t_remaining))
        
        call = S_cur * np.exp(-q * t_remaining) * norm.cdf(d1) - K * np.exp(-r * t_remaining) * norm.cdf(d1 - sigma * np.sqrt(t_remaining))
        put = K * np.exp(-r * t_remaining) * norm.cdf(-(d1 - sigma * np.sqrt(t_remaining))) - S_cur * np.exp(-q * t_remaining) * norm.cdf(-d1)
        
        delta_call = np.exp(-q * t_remaining) * norm.cdf(d1)
        delta_put = np.exp(-q * t_remaining) * (norm.cdf(d1) - 1)
        gamma = np.exp(-q * t_remaining) * norm.pdf(d1) / (S_cur * sigma * np.sqrt(t_remaining))
        
        return {'straddle': call + put, 'delta': delta_call + delta_put, 'gamma': gamma * 2}
    
    # Initial position
    initial = straddle_value(S, v, T)
    hedge_amount = -initial['delta'] * direction
    
    hedge_pos = -hedge_amount
    cash = -initial['straddle'] * direction + hedge_amount * S
    
    results.append({
        'day': 0, 'S': S,
        'straddle_value': initial['straddle'] * direction,
        'option_pnl': 0.0, 'hedge_pnl': 0.0, 'total_pnl': 0.0,
    })
    
    for day in range(1, min(n_days + 1, int(T * 365))):
        t_remaining = max(T - day / 365, 1/365)
        
        # Simulate Heston price path
        z1, z2 = np.random.normal(size=2)
        dwS = np.sqrt(dt) * z1
        dwV = np.sqrt(dt) * (rho * z1 + np.sqrt(1 - rho**2) * z2)
        
        v = max(v + kappa * (theta - v) * dt + xi * np.sqrt(max(v, 0)) * dwV, 1e-6)
        S_new = max(S + S * (0.15 * dt + np.sqrt(max(v, 0)) * dwS), 1.0)
        
        # Re-price and compute P&L
        current = straddle_value(S_new, v, t_remaining)
        
        option_value_change = (current['straddle'] * direction) - results[-1]['straddle_value']
        hedge_pnl_change = hedge_pos * (S - S_new)
        
        # Update hedge
        new_hedge = -current['delta'] * direction
        hedge_adjustment = new_hedge - hedge_pos
        cash -= hedge_adjustment * S_new
        hedge_pos = new_hedge
        
        total_pnl = option_value_change + hedge_pnl_change
        
        results.append({
            'day': day, 'S': S_new,
            'straddle_value': current['straddle'] * direction,
            'option_pnl': option_value_change,
            'hedge_pnl': hedge_pnl_change,
            'total_pnl': total_pnl,
            'cumulative_pnl': sum([r['total_pnl'] for r in results]) + total_pnl,
        })
        
        S = S_new
    
    return pd.DataFrame(results)


# Run simulation on the first trade
if len(trades) > 0:
    best_trade = trades.iloc[0]
    print(f'Simulating delta-hedged straddle for {best_trade["action"]} at K={best_trade["strike"]}, expiry={best_trade["expiry"]}')
    pnl_df = simulate_delta_hedged_straddle(best_trade, S, r, q, heston_params, n_days=10)
    display(pnl_df)
else:
    print('No trades to simulate. Creating demo ATM straddle...')
    dummy_trade = pd.Series({'strike': 500.0, 'TTE': 0.25, 'direction': 1, 'action': 'BUY'})
    pnl_df = simulate_delta_hedged_straddle(dummy_trade, S, r, q, heston_params, n_days=10)
    display(pnl_df)

## 10. P&L Visualization

In [ ]:
if 'pnl_df' in locals() and len(pnl_df) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Price path
    ax = axes[0, 0]
    ax.plot(pnl_df['day'], pnl_df['S'], 'b-o', markersize=4)
    strike_val = best_trade['strike'] if 'best_trade' in locals() else 500
    ax.axhline(strike_val, color='r', linestyle='--', alpha=0.5, label='Strike')
    ax.set_xlabel('Day'); ax.set_ylabel('Underlying Price ($)')
    ax.set_title('Underlying Price Path'); ax.legend(); ax.grid(True, alpha=0.3)
    
    # Straddle value
    ax = axes[0, 1]
    ax.plot(pnl_df['day'], pnl_df['straddle_value'], 'g-o', markersize=4)
    ax.set_xlabel('Day'); ax.set_ylabel('Straddle Value ($)')
    ax.set_title('Straddle Position Value'); ax.grid(True, alpha=0.3)
    
    # Daily P&L decomposition
    ax = axes[1, 0]
    ax.bar(pnl_df['day'][1:], pnl_df['option_pnl'][1:], alpha=0.7, label='Option P&L', color='steelblue')
    ax.bar(pnl_df['day'][1:], pnl_df['hedge_pnl'][1:], alpha=0.7, label='Hedge P&L', color='coral')
    ax.set_xlabel('Day'); ax.set_ylabel('Daily P&L ($)')
    ax.set_title('Daily P&L Decomposition'); ax.legend(); ax.grid(True, alpha=0.3)
    
    # Cumulative P&L
    ax = axes[1, 1]
    ax.plot(pnl_df['day'], pnl_df['cumulative_pnl'], 'b-o', markersize=4, linewidth=2)
    ax.axhline(0, color='gray', linestyle='-', alpha=0.3)
    ax.set_xlabel('Day'); ax.set_ylabel('Cumulative P&L ($)')
    ax.set_title('Cumulative P&L'); ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print('=' * 60)
    print('P&L ATTRIBUTION SUMMARY')
    print('=' * 60)
    print(f'Total option P&L: ${pnl_df["option_pnl"].sum():.4f}')
    print(f'Total hedge P&L: ${pnl_df["hedge_pnl"].sum():.4f}')
    print(f'Net P&L: ${pnl_df["cumulative_pnl"].iloc[-1]:.4f}')

---
## Summary

### What This Notebook Achieved
1. **Data Pipeline**: Downloaded (or generated) options chain data
2. **Surface Construction**: Built a clean OTM IV surface across multiple expiries
3. **Heston Calibration**: Applied differential evolution + local refinement
4. **Mispricing Detection**: Identified strikes where market IV deviates from Heston fair value
5. **Strategy Generation**: Built delta-hedged straddle positions
6. **Backtest & P&L**: Simulated daily delta-hedging with P&L decomposition

### Next Steps
- Add transaction costs (bid-ask spreads)
- Use full Heston Greeks (bump and recompute)
- Implement rolling calibration window
- Paper trade before live capital